# My Content — abrir en Google Colab
Libreta en español para [mateoperez08251-crypto/my-content](https://github.com/mateoperez08251-crypto/my-content).

**Inicio:** ejecuta las celdas 1–6 en orden para preparar el doblaje por hablantes. La celda 5 descarga los modelos; puedes omitirla si solo usarás voz única. La interfaz aparece dentro de la libreta.

La instalación básica abre la app y prepara Smart Split con transcripción Groq y voz única Edge TTS. La generación de video, Whisper local y clonación por hablantes requieren modelos y recursos adicionales; abrir la interfaz no significa que todos los modelos estén instalados.

**Colab:** usar principalmente una interfaz web en un entorno gratuito sin saldo de unidades tiene restricciones. Para ese uso necesitas un plan con saldo positivo o un entorno local compatible; no se incluyen trucos para evitar desconexiones. [Normas y límites oficiales](https://research.google.com/colaboratory/faq.html). El tiempo, la GPU y la memoria no están garantizados.

Antes de empezar: Entorno de ejecución → Cambiar tipo de entorno → GPU, si usarás modelos locales. Los archivos de `/content` se pierden al eliminar la sesión: descarga tus resultados con la celda 8.

Revisión del repositorio: `main`. Incluye descarga de modelos, voces por persona y exportación rápida. No modifica ni publica cambios en GitHub.

**Corrección de instalación:** se comprueba Python y pip, se conserva cualquier entorno incompleto y se usa virtualenv si falla venv/ensurepip. Si ya ejecutaste las celdas 1 y 2 en esta sesión, reemplaza solo la celda 3 por la nueva y ejecútala.


In [ ]:
# 1. Configuración
from pathlib import Path
import os, sys, subprocess, json, time, socket, signal, getpass, shutil

REPO_URL = "https://github.com/mateoperez08251-crypto/my-content.git"
REVISION = "main"  # También puedes fijar un commit concreto.
ROOT = Path("/content/my-content-colab")
APP = ROOT / "repo"
DATA = ROOT / "datos"
VENV = ROOT / "venv"
PYTHON = str(VENV / "bin/python")
PORT = 5001
ROOT.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

def run(args, **kwargs):
    return subprocess.run([str(x) for x in args], check=True, **kwargs)

def stop_tunnel():
    global tunnel_process
    proc = globals().get("tunnel_process")
    if proc is not None and proc.poll() is None:
        os.killpg(proc.pid, signal.SIGTERM)
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait(timeout=10)
    tunnel_process = None

def stop_app():
    global app_process
    stop_tunnel()
    proc = globals().get("app_process")
    if proc is not None and proc.poll() is None:
        os.killpg(proc.pid, signal.SIGTERM)
        try:
            proc.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait(timeout=10)
    app_process = None

print("Carpeta de trabajo:", ROOT)
if shutil.which("nvidia-smi"):
    run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
else:
    print("Sin GPU: la interfaz y Groq funcionan; los modelos locales pueden ser lentos o no caber.")


In [ ]:
# 2. Descargar el repositorio sin borrar cambios locales
stop_app()
if not (APP / ".git").exists():
    run(["git", "clone", REPO_URL, APP])
else:
    dirty = subprocess.check_output(["git", "-C", str(APP), "status", "--porcelain"], text=True)
    if dirty.strip():
        raise RuntimeError("Hay cambios locales en el repositorio. Guárdalos antes de actualizar; no se borrarán.")
    run(["git", "-C", APP, "fetch", "origin", "main"])
target = "origin/main" if REVISION == "main" else REVISION
run(["git", "-C", APP, "checkout", "--detach", target])
print("Revisión:", subprocess.check_output(["git", "-C", str(APP), "rev-parse", "HEAD"], text=True).strip())
assert (APP / "content.py").is_file(), "No se encontró el programa."


In [ ]:
# 3. Instalar la app para Linux (sin ventana de Windows)
stop_app()
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "ffmpeg", "libgl1", "libglib2.0-0", "fonts-noto-cjk", "python3-venv"])
def ensure_environment(folder, system_packages=False):
    """Comprueba Python y pip; recupera instalaciones parciales sin borrar archivos."""
    folder = Path(folder)
    python = folder / "bin/python"
    def valid():
        if not python.exists():
            return False
        check = subprocess.run([str(python), "-c",
            "import sys, pip; assert sys.prefix != sys.base_prefix"],
            capture_output=True, text=True)
        return check.returncode == 0
    if valid():
        return str(python)
    def preserve_partial():
        if folder.exists():
            import uuid
            backup = folder.with_name(folder.name + ".incompleto-" + uuid.uuid4().hex[:8])
            folder.rename(backup)
            print("Entorno incompleto conservado en", backup)
    preserve_partial()
    command = [sys.executable, "-m", "venv"]
    if system_packages:
        command.append("--system-site-packages")
    result = subprocess.run(command + [str(folder)], capture_output=True, text=True)
    if result.returncode == 0 and valid():
        return str(python)
    print("venv no terminó correctamente. Detalle:")
    print((result.stdout + result.stderr)[-6000:])
    print("Reintentando con virtualenv, que instala pip sin depender de ensurepip...")
    preserve_partial()
    run(["apt-get", "install", "-y", "-qq", "python3-virtualenv"])
    executable = shutil.which("virtualenv")
    if not executable:
        raise RuntimeError("No se encontró virtualenv después de instalarlo.")
    command = [executable, "--python", sys.executable]
    if system_packages:
        command.append("--system-site-packages")
    result = subprocess.run(command + [str(folder)], capture_output=True, text=True)
    if result.returncode or not valid():
        print((result.stdout + result.stderr)[-6000:])
        raise RuntimeError("No se pudo crear el entorno. Comparte el detalle anterior a este error.")
    return str(python)

PYTHON = ensure_environment(VENV, system_packages=True)
run([PYTHON, "-m", "pip", "install", "--upgrade", "pip"])
packages = [
    "flask>=3,<4", "requests>=2.31", "pillow>=10.1", "numpy", "imageio-ffmpeg>=0.5",
    "moviepy>=2,<3", "opencv-python-headless>=4.8", "yt-dlp[default,deno]",
    "edge-tts>=7.2,<8", "huggingface_hub", "psutil", "pyperclip",
]
run([PYTHON, "-m", "pip", "install", *packages])
run([PYTHON, "-c", "import flask, requests, moviepy, cv2, edge_tts; print('Dependencias básicas: OK')"])
print("Instalación básica terminada. No hace falta instalar pywebview, pystray ni pyautogui para abrir la interfaz.")


## Claves privadas
En el panel de llave 🔑 de Colab puedes añadir `GROQ_API_KEY`, `HF_TOKEN` y `CONTENTAPP_CLAVE`, y conceder acceso a esta libreta. También puedes introducirlas en los campos ocultos que aparecen al ejecutar la siguiente celda.

Groq es necesario para la selección de clips, traducción y la transcripción en nube. Hugging Face solo se necesita para modelos que requieran acceso. La contraseña protege la app; el usuario de acceso puede ser `colab`. No escribas claves directamente en las celdas ni compartas registros sin revisarlos.


In [ ]:
# 4. Preparar entorno y claves (no se imprimen)
def secret(name, prompt):
    value = os.environ.get(name, "")
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = ""
    return value or getpass.getpass(prompt)

app_env = os.environ.copy()
app_env.update({
    "CONTENTAPP_SERVIDOR": "1", "CONTENTAPP_DATA_DIR": str(DATA),
    "CONTENTAPP_PUERTO": str(PORT), "CONTENTAPP_VIDEO_PYTHON": PYTHON,
    "CONTENTAPP_AUTOBORRAR_MIN": "0", "CONTENTAPP_MOTOR_PERSISTENTE": "0",
    "HF_HOME": str(DATA / "huggingface"),
    "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8",
    "SMART_SPLIT_CJK_FONT": "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
})
app_env["CONTENTAPP_CLAVE"] = secret("CONTENTAPP_CLAVE", "Contraseña para abrir la app: ")
if not app_env["CONTENTAPP_CLAVE"].strip():
    raise ValueError("Escribe una contraseña y repite esta celda.")
for name, label in [("GROQ_API_KEY", "Clave de Groq (Enter para configurar después): "),
                    ("HF_TOKEN", "Token de Hugging Face (opcional, Enter para omitir): ")]:
    value = secret(name, label)
    if value:
        app_env[name] = value
print("Entorno preparado. Recuerda la contraseña que acabas de introducir.")


## 5. Preparar doblaje por personas y descargar modelos
Esta celda instala **VoxCPM2** y el detector **pyannote Community-1** y descarga ambos. VoxCPM2 es el modelo multilingüe elegido para esta app; no hace falta buscarlo en otro menú. Reserva varios GB de disco y tiempo para la primera instalación. Se usa Python 3.12 para los motores porque VoxCPM2 no admite Python 3.13.

Antes de ejecutar: añade `HF_TOKEN` en la celda 4 y acepta personalmente las [condiciones del detector](https://huggingface.co/pyannote/speaker-diarization-community-1). Si no aceptas, Hugging Face bloqueará su descarga. Pon `PREPARAR_DOBLAJE = False` para usar solo Groq y voz única.

En Smart Split: elige idioma → activa hablantes → **Detectar y escuchar hablantes** → elige la voz para cada persona. También puedes descargar VoxCPM2 con el botón dentro de Smart Split. Usa voces propias o autorizadas.


In [ ]:
# 5. Instalar motores y descargar voz + detector (primera ejecución: varios GB)
PREPARAR_DOBLAJE = True

if PREPARAR_DOBLAJE:
    stop_app()
    if not app_env.get("HF_TOKEN"):
        raise ValueError("Configura HF_TOKEN en la celda 4 y acepta las condiciones del detector antes de seguir.")
    # Entornos aislados: Colab puede usar Python 3.13, incompatible con VoxCPM2.
    run([PYTHON, "-m", "pip", "install", "uv"])
    voice_env = ROOT / "venv-motores-312"
    diar_env = ROOT / "venv-diarizacion-312"
    for target in (voice_env, diar_env):
        binary = target / "bin/python"
        if not binary.exists():
            run([PYTHON, "-m", "uv", "venv", "--python", "3.12", "--seed", target])
        run([binary, "-m", "pip", "--version"])
    voice_python = str(voice_env / "bin/python")
    diar_python = str(diar_env / "bin/python")
    run([voice_python, "-m", "pip", "install", "torch>=2.5", "torchaudio"])
    run([voice_python, "-m", "pip", "install", "-r", APP / "runpod/requirements-runpod.txt", "faster-whisper", "soundfile", "librosa"])
    run([voice_python, APP / "tts_worker.py", "--instalar"], cwd=APP, env=app_env)
    run([voice_python, APP / "video_worker.py", "--diagnostico"], cwd=APP, env=app_env)
    run([diar_python, "-m", "pip", "install", "-r", APP / "requirements-smart-speakers.txt"])
    app_env["CONTENTAPP_VIDEO_PYTHON"] = voice_python
    app_env["SMART_DIARIZATION_PYTHON"] = diar_python
    print("Descargando VoxCPM2 al directorio de modelos de la app…", flush=True)
    run([voice_python, "-c", r"""
import os
from pathlib import Path
from huggingface_hub import snapshot_download
folder = Path(os.environ['CONTENTAPP_DATA_DIR']) / 'models' / 'video_ai' / 'voxcpm2'
snapshot_download('openbmb/VoxCPM2', local_dir=folder,
    allow_patterns=['config.json', 'model.safetensors', 'audiovae.pth', 'tokenizer.json',
                    'tokenizer_config.json', 'special_tokens_map.json', 'tokenization_voxcpm2.py'])
for name in ['config.json', 'model.safetensors', 'audiovae.pth', 'tokenizer.json']:
    if not (folder / name).is_file() or (folder / name).stat().st_size == 0:
        raise RuntimeError('Descarga incompleta: ' + name)
from tts_worker import cargar_modelo
model = cargar_modelo(str(folder))
(folder / '.completo').write_text('VoxCPM2 descargado y cargado correctamente', encoding='utf-8')
print('VoxCPM2 listo.', flush=True)
"""], cwd=APP, env=app_env)
    print("Descargando y comprobando el detector…", flush=True)
    run([diar_python, "-c", r"""
import os
from pyannote.audio import Pipeline
pipeline = Pipeline.from_pretrained('pyannote/speaker-diarization-community-1', token=os.environ['HF_TOKEN'])
if pipeline is None:
    raise RuntimeError('Acepta las condiciones del modelo y comprueba HF_TOKEN.')
print('Detector listo.', flush=True)
"""], cwd=APP, env=app_env)
    print("Voz y detector listos. Ejecuta la celda 6 para abrir la app.")
else:
    print("Doblaje local omitido. Puedes abrir la app con Groq y voz única.")


In [ ]:
# 6. Arrancar y abrir la interfaz dentro de Colab
from google.colab import output
import urllib.request, urllib.error, base64
stop_app()
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError(f"El puerto {PORT} está ocupado por otro proceso; cambia PORT en la celda 1 y repite la 4.")
log_path = ROOT / "servidor.log"
with log_path.open("w", encoding="utf-8") as log:
    app_process = subprocess.Popen([PYTHON, str(APP / "content.py"), "--servidor"],
        cwd=APP, env=app_env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)

def redacted_log():
    text = log_path.read_text(encoding="utf-8", errors="replace")[-12000:]
    for key in ("CONTENTAPP_CLAVE", "GROQ_API_KEY", "HF_TOKEN"):
        if app_env.get(key): text = text.replace(app_env[key], "[OCULTO]")
    return text

ready = False
for attempt in range(60):
    if app_process.poll() is not None: break
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/api/health")
        credentials = base64.b64encode(("colab:" + app_env["CONTENTAPP_CLAVE"]).encode()).decode()
        req.add_header("Authorization", "Basic " + credentials)
        with urllib.request.urlopen(req, timeout=2) as response:
            ready = response.status == 200
        if ready: break
    except (OSError, urllib.error.URLError):
        time.sleep(1)
if not ready:
    stop_app()
    print(redacted_log())
    raise RuntimeError("La app no arrancó; revisa el error anterior.")
print("App iniciada. Si pide acceso: usuario colab y tu contraseña de la celda 4.")
output.serve_kernel_port_as_iframe(PORT, height=900, cache_in_notebook=False)


## 6B. Abrir en otra ventana
La celda siguiente crea una URL HTTPS con ngrok y contraseña. Conserva la interfaz dentro de Colab. Crea una cuenta en [ngrok](https://dashboard.ngrok.com/signup) y copia tu [authtoken](https://dashboard.ngrok.com/get-started/your-authtoken) al secreto `NGROK_AUTHTOKEN` de Colab o al campo oculto. Ngrok procesa el tráfico del enlace y aplica los límites de tu cuenta. La URL deja de funcionar al detener el proceso o cerrar el entorno; no mantiene Colab conectado. Siguen aplicándose las restricciones de Colab indicadas al principio.


In [ ]:
# 6B. URL HTTPS para abrir la app en otra ventana (opcional)
# Requiere una cuenta ngrok. El enlace funciona mientras Colab y la app estén activos.
from IPython.display import display, HTML
import html, urllib.request, urllib.error, base64

if globals().get("app_process") is None or app_process.poll() is not None:
    raise RuntimeError("Primero inicia la app con la celda 6.")
stop_tunnel()
token = secret("NGROK_AUTHTOKEN", "Authtoken de ngrok (entrada oculta): ").strip()
if not token:
    raise ValueError("Añade NGROK_AUTHTOKEN a los secretos de Colab o introdúcelo al ejecutar esta celda.")
run([PYTHON, "-m", "pip", "install", "pyngrok>=8.1,<9"])
url_file = ROOT / "url_externa.json"
url_file.unlink(missing_ok=True)
worker_file = ROOT / "abrir_url.py"
worker_file.write_text('import json, os, signal, time\nfrom pathlib import Path\nfrom pyngrok import conf, ngrok\ncfg = conf.PyngrokConfig(auth_token=os.environ["NGROK_AUTHTOKEN"],\n    config_path=os.environ["COLAB_NGROK_CONFIG"], config_version="2")\ntry:\n    tunnel = ngrok.connect(addr="127.0.0.1:" + os.environ["CONTENTAPP_PUERTO"], proto="http",\n        bind_tls=True, basic_auth=["colab:" + os.environ["CONTENTAPP_CLAVE"]],\n        inspect=False, pyngrok_config=cfg)\n    Path(os.environ["COLAB_URL_FILE"]).write_text(json.dumps({"url":tunnel.public_url}))\n    while True:\n        time.sleep(1)\nfinally:\n    ngrok.kill(pyngrok_config=cfg)\n', encoding="utf-8")
tunnel_env = dict(app_env, NGROK_AUTHTOKEN=token,
    COLAB_NGROK_CONFIG=str(ROOT / "ngrok.yml"), COLAB_URL_FILE=str(url_file))
with (ROOT / "tunel.log").open("w", encoding="utf-8") as log:
    tunnel_process = subprocess.Popen([PYTHON, str(worker_file)], env=tunnel_env,
        stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
del token, tunnel_env
try:
    for _ in range(45):
        if tunnel_process.poll() is not None:
            raise RuntimeError("Ngrok no pudo iniciar. Revisa el authtoken, el estado de tu cuenta y sus límites. El registro está en tunel.log; no lo compartas sin revisar.")
        if url_file.exists(): break
        time.sleep(1)
    else:
        raise RuntimeError("Ngrok no respondió a tiempo. Vuelve a ejecutar esta celda.")
    public_url = json.loads(url_file.read_text())["url"]
    if not public_url.startswith("https://"):
        raise RuntimeError("No se recibió una URL HTTPS.")
    headers = {"ngrok-skip-browser-warning":"1"}
    req = urllib.request.Request(public_url + "/api/health", headers=headers)
    try:
        urllib.request.urlopen(req, timeout=20).close()
        raise RuntimeError("El enlace no pidió contraseña. Se cerrará por seguridad.")
    except urllib.error.HTTPError as exc:
        if exc.code != 401: raise RuntimeError(f"El túnel respondió HTTP {exc.code}.") from None
    credentials = base64.b64encode(("colab:" + app_env["CONTENTAPP_CLAVE"]).encode()).decode()
    req.add_header("Authorization", "Basic " + credentials)
    with urllib.request.urlopen(req, timeout=20) as response:
        if response.status != 200: raise RuntimeError("La app no responde por el enlace.")
    display(HTML('<a href="' + html.escape(public_url, quote=True) + '" target="_blank" rel="noopener noreferrer">Abrir My Content en otra ventana</a>'))
    print("URL:", public_url)
    print("Usuario: colab · Contraseña: la de la celda 4.")
    print("Mantén Colab activo. Para cerrar la URL y la app, ejecuta la celda 9.")
except Exception:
    stop_tunnel()
    raise


## Uso y límites
- En Smart Split, sube el video con el selector de archivos del navegador. Los archivos están en Colab, no en tu PC.
- Para comenzar, usa **Groq** como transcripción y **voz única**. Prueba primero un clip corto.
- Puedes ver toda la interfaz, pero la publicación automatizada en redes, los diálogos de Windows y los modelos no instalados no están garantizados en Colab. El inicio de sesión en YouTube/TikTok puede requerir configuración adicional.
- Si la interfaz queda en blanco, permite las cookies de `googleusercontent.com`, vuelve a ejecutar la celda 6 y revisa la celda 7. No se usa `serve_kernel_port_as_window`, que Colab marca como obsoleto por cambios del navegador.
- La URL opcional de ngrok es accesible desde internet y exige contraseña; no compartas sus credenciales. No hay bucles para impedir que Colab cierre la sesión.


In [ ]:
# 7. Diagnóstico / volver a mostrar interfaz
if "app_process" not in globals() or app_process is None:
    print("Ejecuta la celda 6.")
else:
    print("Servidor:", "activo" if app_process.poll() is None else "detenido")
    print(redacted_log())
    if app_process.poll() is None:
        output.serve_kernel_port_as_iframe(PORT, height=900, cache_in_notebook=False)


In [ ]:
# 8. Descargar resultados (no incluye claves, cookies, registros ni modelos)
from google.colab import files
import zipfile
RESULTADOS = DATA / "videos_procesados"  # Cambia esta carpeta si elegiste otra al exportar.
archive = ROOT / "resultados_my_content.zip"
allowed = {".mp4", ".mov", ".webm", ".wav", ".mp3", ".srt", ".vtt", ".png", ".jpg"}
media = [p for p in RESULTADOS.rglob("*") if p.is_file() and not p.is_symlink() and p.suffix.lower() in allowed] if RESULTADOS.exists() else []
if not media:
    print("Todavía no hay resultados en", RESULTADOS)
else:
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_STORED) as z:
        for p in media:
            z.write(p, p.relative_to(RESULTADOS))
    print(f"Preparados {len(media)} archivos. Para videos grandes usa el panel Archivos de Colab.")
    files.download(str(archive))


In [ ]:
# 9. Detener la app y sus procesos hijos cuando termines
stop_app()
print("App detenida. Los archivos siguen en esta sesión; descárgalos antes de eliminar el entorno.")


## Fuentes y validación
- [Repositorio utilizado](https://github.com/mateoperez08251-crypto/my-content/tree/main)
- [Interfaz iframe oficial de Colab](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py)
- [Límites de Colab](https://research.google.com/colaboratory/faq.html)

La estructura de la libreta y la sintaxis de las celdas se verificaron fuera de Colab. No se ha ejecutado una sesión real de Colab ni probado la descarga de todos los modelos. Si una instalación falla, conserva el mensaje y evita ejecutar las siguientes celdas hasta resolverla.
